# Mechanistic Synthesis: PRMP Improvement Decomposition, Gradient/Embedding Analysis

**What this notebook does:**  
Synthesizes evidence from 4 prior PRMP experiments into a unified mechanistic evaluation:

1. **Mechanism Decomposition** — Shows 73.1% of PRMP's improvement is irreducible to extra parameters, MLP, or skip connections (Amazon RMSE)
2. **Gradient Norm Ratio Analysis** — Mann-Whitney test (p<0.0001) comparing regression vs classification tasks
3. **Embedding Effective Rank Analysis** — Spearman correlation (ρ=-0.700) between rank change and improvement
4. **Embedding R² Correlation** — Higher R² predicts larger per-link improvement
5. **Cross-Experiment Summary** — Cohen's d=1.663 for regression vs classification effect

Produces 4 publication-ready figures and comprehensive statistical analysis.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'matplotlib==3.10.0')

In [ ]:
import json
import math
import os

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
from scipy import stats

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-b2d5b0-predictive-residual-message-passing-filt/main/evaluation_iter5_mechanistic_syn/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded data with keys: {list(data.keys())}")

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Tunable parameters — start with minimum values, scale up if time permits

N_BOOT = 100  # Bootstrap iterations (original: 10000)

## Helper Functions

Bootstrap confidence interval estimation and Cohen's d effect size computation.

In [ ]:
def bootstrap_ci(values, n_boot=None, alpha=0.05):
    """Bootstrap 95% CI for the mean of values."""
    if n_boot is None:
        n_boot = N_BOOT
    arr = np.array(values)
    if len(arr) < 2:
        return (float(arr[0]), float(arr[0]))
    rng = np.random.default_rng(42)
    means = np.array([rng.choice(arr, size=len(arr), replace=True).mean() for _ in range(n_boot)])
    lo = float(np.percentile(means, 100 * alpha / 2))
    hi = float(np.percentile(means, 100 * (1 - alpha / 2)))
    return (lo, hi)


def cohens_d(group1, group2):
    """Compute Cohen's d (positive = group1 > group2)."""
    m1, m2 = np.mean(group1), np.mean(group2)
    n1, n2 = len(group1), len(group2)
    s1, s2 = np.std(group1, ddof=1), np.std(group2, ddof=1)
    pooled = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2)) if (n1 + n2 > 2) else 1e-9
    if pooled < 1e-12:
        pooled = 1e-12
    return float((m1 - m2) / pooled)

print("Helper functions defined.")

## Metric 1: Mechanism Decomposition (Amazon Video Games)

Decomposes PRMP's improvement into contributions from extra parameters, auxiliary MLP, skip connections, and the irreducible predict-subtract residual. Uses bootstrap CIs on per-seed RMSE/MAE/R² values.

In [ ]:
summaries = data["amazon_summaries"]
decomp = {}

for metric_name in ["rmse", "mae", "r2"]:
    std_vals = summaries["A_standard_sage"][metric_name]["values"]
    prmp_vals = summaries["B_prmp"][metric_name]["values"]
    wide_vals = summaries["C_wide_sage"][metric_name]["values"]
    aux_vals = summaries["D_aux_mlp"][metric_name]["values"]
    skip_vals = summaries["E_skip_residual"][metric_name]["values"]

    std_mean = np.mean(std_vals)
    prmp_mean = np.mean(prmp_vals)
    total_improvement = std_mean - prmp_mean  # positive for RMSE/MAE
    if metric_name == "r2":
        total_improvement = prmp_mean - std_mean  # positive for R²

    if abs(total_improvement) < 1e-12:
        print(f"No improvement for {metric_name}, skipping")
        continue

    wide_mean = np.mean(wide_vals)
    aux_mean = np.mean(aux_vals)
    skip_mean = np.mean(skip_vals)

    if metric_name == "r2":
        extra_params_frac = (wide_mean - std_mean) / total_improvement
        aux_mlp_frac = (aux_mean - std_mean) / total_improvement
        skip_frac = (skip_mean - std_mean) / total_improvement
    else:
        extra_params_frac = (std_mean - wide_mean) / total_improvement
        aux_mlp_frac = (std_mean - aux_mean) / total_improvement
        skip_frac = (std_mean - skip_mean) / total_improvement

    residual_frac = 1.0 - max(extra_params_frac, aux_mlp_frac, skip_frac)

    # Bootstrap CIs for each fraction
    rng = np.random.default_rng(42)
    frac_boots = {"extra_params": [], "aux_mlp": [], "skip": [], "residual": []}
    for _ in range(N_BOOT):
        idx = rng.integers(0, len(std_vals), size=len(std_vals))
        s = np.mean([std_vals[i] for i in idx])
        p = np.mean([prmp_vals[i] for i in idx])
        w = np.mean([wide_vals[i] for i in idx])
        a = np.mean([aux_vals[i] for i in idx])
        sk = np.mean([skip_vals[i] for i in idx])
        ti = (p - s) if metric_name == "r2" else (s - p)
        if abs(ti) < 1e-12:
            continue
        if metric_name == "r2":
            ep = (w - s) / ti; am = (a - s) / ti; sf = (sk - s) / ti
        else:
            ep = (s - w) / ti; am = (s - a) / ti; sf = (s - sk) / ti
        rf = 1.0 - max(ep, am, sf)
        frac_boots["extra_params"].append(ep)
        frac_boots["aux_mlp"].append(am)
        frac_boots["skip"].append(sf)
        frac_boots["residual"].append(rf)

    ci = {}
    for k, v in frac_boots.items():
        ci[k] = (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))) if v else (0.0, 0.0)

    decomp[metric_name] = {
        "total_improvement": float(total_improvement),
        "standard_mean": float(std_mean),
        "prmp_mean": float(prmp_mean),
        "extra_params_frac": float(extra_params_frac),
        "extra_params_ci": ci["extra_params"],
        "aux_mlp_frac": float(aux_mlp_frac),
        "aux_mlp_ci": ci["aux_mlp"],
        "skip_frac": float(skip_frac),
        "skip_ci": ci["skip"],
        "predict_subtract_residual_frac": float(residual_frac),
        "predict_subtract_ci": ci["residual"],
        "variant_means": {
            "standard": float(std_mean), "wide": float(wide_mean),
            "aux_mlp": float(aux_mean), "skip_residual": float(skip_mean),
            "prmp": float(prmp_mean),
        },
    }
    print(f"  {metric_name}: extra_params={extra_params_frac:.3f}, aux_mlp={aux_mlp_frac:.3f}, "
          f"skip={skip_frac:.3f}, residual={residual_frac:.3f}")

### Figure 1: Mechanism Decomposition Bar Chart

Compares Amazon RMSE across model variants: Standard, Wide (+params), AuxMLP (+MLP), Skip-Residual (+shortcut), and PRMP (predict-subtract). Annotates the fraction of improvement attributable to each mechanism.

In [ ]:
variants = ["A_standard_sage", "C_wide_sage", "D_aux_mlp", "E_skip_residual", "B_prmp"]
labels = ["Standard", "Wide\n(+params)", "AuxMLP\n(+MLP)", "Skip-Res\n(+shortcut)", "PRMP\n(predict-subtract)"]
colors = ["#7f7f7f", "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(variants))
means = [np.mean(summaries[v]["rmse"]["values"]) for v in variants]
stds = [np.std(summaries[v]["rmse"]["values"]) for v in variants]

bars = ax.bar(x, means, yerr=stds, capsize=5, color=colors, edgecolor="black", linewidth=0.5, width=0.65)

# Dashed lines
std_mean = means[0]
prmp_mean = means[-1]
ax.axhline(std_mean, color="#7f7f7f", linestyle="--", alpha=0.6, linewidth=1)
ax.axhline(prmp_mean, color="#d62728", linestyle="--", alpha=0.6, linewidth=1)

# Annotate fractions
rmse_decomp = decomp.get("rmse", {})
fracs = {
    "Wide": rmse_decomp.get("extra_params_frac", 0),
    "AuxMLP": rmse_decomp.get("aux_mlp_frac", 0),
    "Skip-Res": rmse_decomp.get("skip_frac", 0),
}
for i, (lbl, frac) in enumerate(fracs.items(), start=1):
    ax.annotate(f"{frac:.0%}", xy=(x[i], means[i]), xytext=(x[i], means[i] - 0.012),
                ha="center", va="top", fontsize=9, fontweight="bold", color=colors[i])

# Residual annotation
res_frac = rmse_decomp.get("predict_subtract_residual_frac", 0)
ax.annotate(f"Residual: {res_frac:.0%}", xy=(x[-1], prmp_mean),
            xytext=(x[-1] + 0.35, prmp_mean + 0.015), ha="left", va="bottom",
            fontsize=9, fontweight="bold", color="#d62728",
            arrowprops=dict(arrowstyle="->", color="#d62728", lw=1.2))

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel("RMSE (lower is better)", fontsize=11)
ax.set_title("Amazon Video Games: Mechanism Decomposition", fontsize=13, fontweight="bold")
ax.set_ylim(0.48, 0.60)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
plt.show()

## Metric 2 + Figure 2: Gradient Norm Ratio Analysis

Analyzes the ratio of update_mlp to pred_mlp gradient norms across tasks. Mann-Whitney U test compares regression vs classification gradient distributions. The figure shows gradient ratio trajectories over training epochs.

In [ ]:
grad_result = data["gradient_analysis"]

print(f"Regression mean gradient ratio: {grad_result['regression_mean_ratio']:.3f} (n={grad_result['regression_n_edges']})")
print(f"Classification mean gradient ratio: {grad_result['classification_mean_ratio']:.3f} (n={grad_result['classification_n_edges']})")
print(f"Mann-Whitney U: {grad_result['mann_whitney_U']:.1f}, p={grad_result['mann_whitney_p']:.6f}")

# Plot gradient ratio trajectories
trajectories = grad_result["epoch_trajectories"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
reg_tasks = [t for t, d in trajectories.items() if d["task_type"] == "regression"]
cls_tasks = [t for t, d in trajectories.items() if d["task_type"] == "classification"]

line_styles = ["-", "--", ":", "-."]
markers = ["o", "s", "^", "D", "v"]
blue_palette = ["#1f77b4", "#4a90d9"]
red_palette = ["#d62728", "#e8775a", "#c44e52"]

for i, task in enumerate(sorted(reg_tasks)):
    d = trajectories[task]
    axes[0].plot(d["epochs"], d["mean_ratio"], linestyle=line_styles[i % len(line_styles)],
                 marker=markers[i % len(markers)], color=blue_palette[i % len(blue_palette)],
                 label=task, linewidth=2, markersize=5)
    if d["std_ratio"]:
        m = np.array(d["mean_ratio"]); s = np.array(d["std_ratio"])
        axes[0].fill_between(d["epochs"], m - s, m + s, alpha=0.15,
                             color=blue_palette[i % len(blue_palette)])

for i, task in enumerate(sorted(cls_tasks)):
    d = trajectories[task]
    axes[1].plot(d["epochs"], d["mean_ratio"], linestyle=line_styles[i % len(line_styles)],
                 marker=markers[i % len(markers)], color=red_palette[i % len(red_palette)],
                 label=task, linewidth=2, markersize=5)
    if d["std_ratio"]:
        m = np.array(d["mean_ratio"]); s = np.array(d["std_ratio"])
        axes[1].fill_between(d["epochs"], m - s, m + s, alpha=0.15,
                             color=red_palette[i % len(red_palette)])

axes[0].set_title("Regression Tasks", fontsize=12, fontweight="bold", color="#1f77b4")
axes[1].set_title("Classification Tasks", fontsize=12, fontweight="bold", color="#d62728")
for ax in axes:
    ax.set_xlabel("Epoch", fontsize=11)
    ax.legend(fontsize=9, loc="best")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
axes[0].set_ylabel("Gradient Norm Ratio (update_mlp / pred_mlp)", fontsize=10)

fig.suptitle("Gradient Routing: update_mlp vs pred_mlp by Task Type", fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

## Metric 3 + Figure 3: Embedding Effective Rank Analysis

Compares embedding effective rank between Standard and PRMP models across tasks. Computes Spearman correlation between delta rank and performance improvement. Classification tasks show larger rank increases than regression tasks.

In [ ]:
rank_result = data["embedding_rank_analysis"]

# Spearman correlation between delta_rank and delta improvement
delta_ranks = rank_result["delta_ranks"]
task_deltas = rank_result["task_deltas"]
common_tasks = sorted(set(delta_ranks.keys()) & set(task_deltas.keys()))

if len(common_tasks) >= 3:
    dr = [delta_ranks[t] for t in common_tasks]
    dd = [task_deltas[t] for t in common_tasks]
    res = stats.spearmanr(dr, dd)
    print(f"Spearman rho={res.correlation:.3f}, p={res.pvalue:.4f}")
else:
    print("Not enough common tasks for Spearman correlation")

print(f"Regression mean delta rank: {rank_result['regression_mean_delta_rank']:.4f}")
print(f"Classification mean delta rank: {rank_result['classification_mean_delta_rank']:.4f}")

# Figure 3: Embedding Effective Rank Comparison
final_ranks = rank_result["final_ranks"]
task_types = rank_result["task_types"]
tasks = sorted(final_ranks.keys())

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(tasks))

for i, task in enumerate(tasks):
    tt = task_types.get(task, "unknown")
    color = "#1f77b4" if tt == "regression" else "#d62728"
    std_rank = final_ranks[task]["standard"]
    prmp_rank = final_ranks[task]["prmp"]

    ax.plot([x[i], x[i]], [std_rank, prmp_rank], color=color, alpha=0.4, linewidth=2)
    ax.scatter(x[i], std_rank, marker="o", s=120, color=color, edgecolors="black",
               linewidths=0.8, zorder=5, label="Standard" if i == 0 else "")
    ax.scatter(x[i], prmp_rank, marker="*", s=200, color=color, edgecolors="black",
               linewidths=0.5, zorder=5, label="PRMP" if i == 0 else "")

    delta = prmp_rank - std_rank
    sign = "+" if delta > 0 else ""
    ax.annotate(f"{sign}{delta:.1f}", xy=(x[i] + 0.15, (std_rank + prmp_rank) / 2),
                fontsize=8, fontweight="bold", color=color)

ax.set_xticks(x)
ax.set_xticklabels(tasks, fontsize=9, rotation=15, ha="right")
ax.set_ylabel("Effective Rank", fontsize=11)
ax.set_title("Embedding Effective Rank: Standard vs PRMP", fontsize=13, fontweight="bold")

legend_elements = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="gray", markersize=10, label="Standard"),
    Line2D([0], [0], marker="*", color="w", markerfacecolor="gray", markersize=14, label="PRMP"),
    Line2D([0], [0], color="#1f77b4", linewidth=3, label="Regression"),
    Line2D([0], [0], color="#d62728", linewidth=3, label="Classification"),
]
ax.legend(handles=legend_elements, fontsize=9, loc="upper left")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
plt.show()

## Metric 4 + Figure 4: Embedding R² Correlation with Per-Link Improvement

Correlates embedding-space predictability (Ridge R²) with per-link PRMP improvement from ablation experiments. Top panel shows R² trajectories over training; bottom panel shows the relationship between R² and RMSE improvement per link type.

In [ ]:
r2_result = data["embedding_r2_correlation"]

print(f"Product R²: std={r2_result['standard_product_r2']:.3f}, prmp={r2_result['prmp_product_r2']:.3f}, "
      f"improvement={r2_result['product_link_improvement']:.4f}")
print(f"Customer R²: std={r2_result['standard_customer_r2']:.3f}, prmp={r2_result['prmp_customer_r2']:.3f}, "
      f"improvement={r2_result['customer_link_improvement']:.4f}")
print(f"R² predicts improvement: {r2_result['r2_predicts_improvement']}")

# Figure 4: R² Trajectories + Scatter
fig, axes = plt.subplots(2, 1, figsize=(10, 8), gridspec_kw={"height_ratios": [1.2, 1]})

# Top panel: R² trajectories
ax = axes[0]
traj = r2_result["r2_trajectories"]
styles = {
    "standard_product_to_review": ("-", "#1f77b4", "Std - Product"),
    "standard_customer_to_review": ("-", "#d62728", "Std - Customer"),
    "prmp_product_to_review": ("--", "#1f77b4", "PRMP - Product"),
    "prmp_customer_to_review": ("--", "#d62728", "PRMP - Customer"),
}
for key, (ls, color, label) in styles.items():
    d = traj[key]
    ax.plot(d["epochs"], d["ridge_r2"], linestyle=ls, color=color, label=label,
            linewidth=2, marker="o" if ls == "-" else "s", markersize=4)
ax.set_xlabel("Epoch", fontsize=10)
ax.set_ylabel("Embedding Ridge R²", fontsize=10)
ax.set_title("Embedding R² Trajectories: Standard vs PRMP", fontsize=12, fontweight="bold")
ax.legend(fontsize=9, ncol=2, loc="lower right")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Bottom panel: scatter
ax2 = axes[1]
r2_vals = [r2_result["standard_product_r2"], r2_result["standard_customer_r2"]]
improvements = [r2_result["product_link_improvement"], r2_result["customer_link_improvement"]]
colors_scatter = ["#1f77b4", "#d62728"]
labels_scatter = ["Product->Review", "Customer->Review"]

for i in range(2):
    ax2.scatter(r2_vals[i], improvements[i], s=200, color=colors_scatter[i],
                edgecolors="black", linewidths=1, zorder=5)
    ax2.annotate(labels_scatter[i], xy=(r2_vals[i], improvements[i]),
                 xytext=(r2_vals[i] + 0.02, improvements[i] + 0.001),
                 fontsize=10, fontweight="bold", color=colors_scatter[i])

ax2.plot(r2_vals, improvements, "--", color="gray", alpha=0.5, linewidth=1.5)
ax2.set_xlabel("Standard Model Embedding R²", fontsize=10)
ax2.set_ylabel("PRMP Per-Link RMSE Improvement", fontsize=10)
ax2.set_title("Higher Embedding R² -> Larger PRMP Benefit", fontsize=12, fontweight="bold")
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

predicts = r2_result["r2_predicts_improvement"]
ax2.text(0.5, 0.05, f"R² predicts per-link improvement: {predicts}",
         transform=ax2.transAxes, fontsize=10, ha="center",
         bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", edgecolor="gray"))

fig.tight_layout()
plt.show()

## Metric 5: Cross-Experiment Summary Statistics

Aggregates PRMP improvement deltas across all experiments. Computes bootstrap CIs for regression and classification groups, and Cohen's d effect size for the regression-vs-classification difference.

In [ ]:
summary = data["cross_experiment_summary"]

reg_deltas = summary["regression_deltas"]["all_values"]
cls_deltas = summary["classification_deltas"]["all_values"]

reg_mean = float(np.mean(reg_deltas))
cls_mean = float(np.mean(cls_deltas))
reg_ci = bootstrap_ci(reg_deltas)
cls_ci = bootstrap_ci(cls_deltas)

# Cohen's d for regression vs classification
effect_size = cohens_d(reg_deltas, cls_deltas) if (reg_deltas and cls_deltas) else float("nan")

print(f"Regression mean delta: {reg_mean:.4f}, CI={reg_ci}")
print(f"Classification mean delta: {cls_mean:.4f}, CI={cls_ci}")
print(f"Cohen's d (regression vs classification): {effect_size:.3f}")

## Results Summary

Key metrics aggregated across all analyses, with a combined visualization of PRMP improvement by task type.

In [ ]:
# ── Results Table ─────────────────────────────────────────────────────────────
rmse_decomp = decomp.get("rmse", {})

print("=" * 70)
print("KEY RESULTS")
print("=" * 70)
print(f"  Predict-subtract residual fraction: {rmse_decomp.get('predict_subtract_residual_frac', 0):.1%}")
print(f"  Gradient MW p-value (reg vs cls):   {grad_result['mann_whitney_p']:.6f}")
print(f"  Rank Spearman rho:                  {rank_result['spearman_rho']:.3f}")
print(f"  R² predicts improvement:            {r2_result['r2_predicts_improvement']}")
print(f"  Regression mean delta:              {reg_mean:.4f}")
print(f"  Classification mean delta:          {cls_mean:.4f}")
print(f"  Reg vs Cls Cohen's d:               {effect_size:.3f}")
print("=" * 70)

# ── Summary Visualization ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Per-task improvement deltas
ax = axes[0]
task_d = summary["task_deltas"]
task_t = summary["task_types"]
task_names = sorted(task_d.keys())
task_vals = [task_d[t] for t in task_names]
task_colors = ["#1f77b4" if task_t.get(t) == "regression" else "#d62728" for t in task_names]

# Add Amazon seeds
amazon_seeds = summary["regression_deltas"]["amazon_per_seed"]
all_names = [f"Amazon s{i}" for i in range(len(amazon_seeds))] + task_names
all_vals = amazon_seeds + task_vals
all_colors = ["#1f77b4"] * len(amazon_seeds) + task_colors

y_pos = np.arange(len(all_names))
bars = ax.barh(y_pos, all_vals, color=all_colors, edgecolor="black", linewidth=0.5, height=0.6)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels(all_names, fontsize=9)
ax.set_xlabel("PRMP Improvement Delta (positive = PRMP better)", fontsize=10)
ax.set_title("Per-Task PRMP Improvement", fontsize=12, fontweight="bold")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Right: Mechanism decomposition pie chart
ax2 = axes[1]
labels_pie = ["Extra Params", "Aux MLP", "Skip Connection", "Predict-Subtract\n(Residual)"]
fracs_pie = [
    max(0, rmse_decomp.get("extra_params_frac", 0)),
    max(0, rmse_decomp.get("aux_mlp_frac", 0)),
    max(0, rmse_decomp.get("skip_frac", 0)),
    max(0, rmse_decomp.get("predict_subtract_residual_frac", 0)),
]
colors_pie = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]
explode = (0, 0, 0, 0.08)
ax2.pie(fracs_pie, explode=explode, labels=labels_pie, colors=colors_pie,
        autopct="%1.0f%%", shadow=False, startangle=140,
        textprops={"fontsize": 10})
ax2.set_title("RMSE Improvement Decomposition", fontsize=12, fontweight="bold")

fig.tight_layout()
plt.show()

print("\nDone! All 4 figures and 5 metrics computed successfully.")